# Sprint 1 service checkpoint

Use this notebook only after completing the Campus practical work in VS Code. It runs your own exported FieldCare source for the checkpoint; it does not provide a worked endpoint or write application code into files.

You will upload the source ZIP created by `python -m tools.checkpoint artifacts/sprint-1-checkpoint.zip`. The setup clones the course repository's `main` branch, restores the submitted source to a fresh temporary folder, and installs the pinned course dependencies. You then inspect your own source, configure credentials privately, and run your own route.

A supported request prepares the document index and calls the real provider, so it uses quota. Invalid input and missing-context checks may stop before generation. Keep the actual output and explain what it proves; this notebook does not grade your work automatically.

In [ ]:
from pathlib import Path
from getpass import getpass
import json
import hashlib
import os
import subprocess
import sys
import tempfile

# Clean up only this notebook's previous child process and temporary workspace.
old_service = globals().get("service")
if old_service is not None:
    old_service.stop()
old_scratch = globals().get("scratch")
if old_scratch is not None:
    old_scratch.cleanup()

scratch = tempfile.TemporaryDirectory(prefix="fieldcare-checkpoint-")
SCRATCH = Path(scratch.name)
REPO = SCRATCH / "course-repo"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", "main",
     "https://github.com/richhiey/ai-app-dev_Mod-B.git", str(REPO)],
    check=True,
)
revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
print("Course source: main", revision[:12])

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO), "-c", str(REPO / "requirements.lock")])

In [ ]:
# Upload the ZIP you exported from your local course repository.
from google.colab import files
from module_b.workspace import WorkspacePolicy, restore_workspace

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one reviewed Sprint 1 checkpoint ZIP.")
archive_name, archive_bytes = next(iter(uploaded.items()))
print("Source archive SHA-256:", hashlib.sha256(archive_bytes).hexdigest())
if len(archive_bytes) > 20 * 1024 * 1024:
    raise ValueError("The checkpoint ZIP exceeds the 20 MiB limit.")
archive_path = SCRATCH / Path(archive_name).name
archive_path.write_bytes(archive_bytes)

CHECKPOINT_POLICY = WorkspacePolicy(
    directory_suffixes={
        "service/fieldcare": (".py",),
        "service/data": (".json",),
        "clients": (".py",),
        "evidence": (".md", ".json"),
    },
    root_files=("requirements.lock", "pyproject.toml"),
)
CHECKPOINT = SCRATCH / "student-source"
restore_workspace(archive_path, CHECKPOINT, policy=CHECKPOINT_POLICY)
print("Restored source files:")
for folder in (CHECKPOINT / "service/fieldcare", CHECKPOINT / "clients", CHECKPOINT / "evidence"):
    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(" -", path.relative_to(CHECKPOINT))
print("\nRestoring validates the archive paths and file types. It does not certify the code or scan file contents for secrets.")

## Inspect your own service before running it

Open the files listed above and confirm the supervisor route, explicit request model, prompt-to-graph binding, model setting, and router registration are present in the source you intend to assess. Compare its existing operations with the new one. If the ZIP contains an incomplete or unintended version, stop here, return to VS Code, and export a fresh archive.

The notebook installs trusted shared helpers from current `main` and loads your `fieldcare` package from the uploaded source folder. It never replaces the source in the clone with generated code. Before continuing, enter the source revision you recorded locally with `git rev-parse HEAD`; this connects your results to the exact working copy you exported.

In [ ]:
# Record the local revision associated with the source archive.
student_revision = input("Repository base revision from `git rev-parse HEAD`: ").strip()
if not student_revision:
    raise ValueError("Enter the repository base revision recorded before exporting the checkpoint.")
print("Repository base revision:", student_revision)

# Load the application package from the uploaded source, not the starter clone.
SERVICE_SOURCE = CHECKPOINT / "service"
sys.path.insert(0, str(SERVICE_SOURCE))
os.environ["FIELDCARE_WORK_DIR"] = str(SCRATCH / "runtime")
os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter key (hidden; used for index + supported request): ").strip()
if not os.environ["OPENROUTER_API_KEY"]:
    raise ValueError("A provider key is required for the supported-answer checkpoint.")

from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
print("Approved course models:", ", ".join(sorted(CHAT_MODELS)))
model_name = input(f"Model for the supervisor route [{DEFAULT_CHAT_MODEL}]: ").strip() or DEFAULT_CHAT_MODEL
if model_name not in CHAT_MODELS:
    raise ValueError("Choose one of the approved course models printed above.")
os.environ["OPENROUTER_MODEL"] = model_name
print("Model setting: OPENROUTER_MODEL =", model_name)

# Index creation sends the supplied synthetic documents to the embedding provider.
from fieldcare.prepare_index import main as prepare_index
prepare_index()

## Start the uploaded application

The checkpoint notebook uses an owned loopback process. It waits for that process's health route and stops only the process it starts. A healthy response tells you the service started; your route requests below provide the contract and behavior evidence.

In [ ]:
from module_b.runtime import ServiceProcess
import httpx

service = ServiceProcess(
    "fieldcare.main:app",
    project_dir=SERVICE_SOURCE,
    env={"FIELDCARE_WORK_DIR": str(SCRATCH / "runtime"), "OPENROUTER_API_KEY": os.environ["OPENROUTER_API_KEY"], "OPENROUTER_MODEL": model_name},
    startup_timeout=120,
).start()
health = httpx.get(f"{service.base_url}/health", timeout=10)
print("Health HTTP status:", health.status_code)
print("Health fields:", health.json())

## Run your checkpoint requests

Enter your versioned supervisor route, POST /v2/diagnose, and a request body that matches your implemented contract. Use synthetic HelioDesk input. The example equipment ID below refers to the supplied synthetic dataset; change it if your checkpoint uses a different record.

First send one supported request. Then send a malformed request that crosses a distinct validation boundary and a request to an earlier route. Keep the actual HTTP status, response fields, request ID, and one cited claim for your handoff. If a model or provider error occurs, report it as observed; do not substitute a sample answer.

In [ ]:
SUPERVISOR_PATH = input("Your supervisor POST path [/v2/diagnose]: ").strip() or "/v2/diagnose"
if SUPERVISOR_PATH != "/v2/diagnose":
    raise ValueError("The Sprint 1 checkpoint expects the versioned supervisor route POST /v2/diagnose.")
body_text = input('JSON body for one supported synthetic supervisor request (example shape: {"question":"Which filter checks are documented?","equipment_id":"EQ-FC-1002"}): ')
supported_body = json.loads(body_text)
if not isinstance(supported_body, dict):
    raise ValueError("The request body must be a JSON object.")

response = httpx.post(service.base_url + SUPERVISOR_PATH, json=supported_body, timeout=180)
request_id = response.headers.get("X-Request-ID")
print("HTTP status:", response.status_code)
print("Request ID:", request_id or "not returned")
try:
    result = response.json()
except ValueError:
    result = response.text
print("Actual response:")
print(json.dumps(result, indent=2) if isinstance(result, (dict, list)) else result)

### Compare validation and compatibility

Use the actual request model in your source to select a malformed input that should fail for a different reason than your supported request. Then check an earlier route using its existing contract. Record your predictions before running each request. Do not treat validation errors as provider failures, or the same HTTP status as proof that two routes share a contract.

In [ ]:
invalid_path = input("Path for the malformed-input check [same as supervisor path]: ").strip() or SUPERVISOR_PATH
invalid_text = input('JSON body that should be rejected by your schema: ').strip()
invalid_body = json.loads(invalid_text)
invalid_response = httpx.post(service.base_url + invalid_path, json=invalid_body, timeout=30)
print("Malformed-input HTTP status:", invalid_response.status_code)
print("Malformed-input response:", invalid_response.text[:2000])

older_path = input("Earlier route path to check compatibility: ").strip()
older_text = input('JSON body for that earlier route: ').strip()
older_body = json.loads(older_text)
older_response = httpx.post(service.base_url + older_path, json=older_body, timeout=180)
print("Earlier-route HTTP status:", older_response.status_code)
print("Earlier-route request ID:", older_response.headers.get("X-Request-ID") or "not returned")
print("Earlier-route response:", older_response.text[:4000])

In [ ]:
# Stop the owned service when you have captured your evidence.
service.stop()
print("Checkpoint service stopped.")

## Handoff

Save the notebook with your actual predictions, repository base revision, source archive SHA-256, routes, statuses, response fields, compatibility result, and one citation check. Explain what the implementation can and cannot know from the supplied documents. Keep API keys out of notebook content and outputs. Pair this notebook with the exact source ZIP that produced the observations.